# 3. Reservoir tables with pandas

        This module replaces the unrelated dataset in the original pandas lecture with a fully
        documented synthetic well table.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [2]:
wells = pd.read_csv(ROOT / "data" / "synthetic_wells.csv")
print(wells.shape)
wells.info()
wells.select_dtypes(include="number").describe().T


(36, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   well_id                    36 non-null     object 
 1   field                      36 non-null     object 
 2   x_m                        36 non-null     float64
 3   y_m                        36 non-null     float64
 4   net_pay_m                  36 non-null     float64
 5   porosity_fraction          36 non-null     float64
 6   water_saturation_fraction  36 non-null     float64
 7   permeability_md            36 non-null     float64
 8   pressure_mpa               36 non-null     float64
 9   temperature_k              36 non-null     float64
 10  oil_rate_bpd               36 non-null     float64
dtypes: float64(9), object(2)
memory usage: 3.2+ KB


,count,mean,std,min,25%,50%,75%,max
x_m,36.0,3408.872222,2243.255768,169.2000,1300.450000,3224.8000,4939.925000,7367.8000
y_m,36.0,2838.925000,1743.733514,260.9000,1414.225000,2639.0500,4674.725000,5877.4000
net_pay_m,36.0,18.499722,4.468442,11.1600,15.895000,18.3850,20.477500,26.6200
porosity_fraction,36.0,0.213078,0.025127,0.1590,0.194625,0.2159,0.229650,0.2850
water_saturation_fraction,36.0,0.414858,0.047705,0.3116,0.375725,0.4122,0.450675,0.5095
permeability_md,36.0,50.743333,15.009223,20.7500,38.455000,48.0400,61.000000,82.2500
pressure_mpa,36.0,23.716111,3.170627,18.0300,21.382500,23.1450,25.315000,32.1600
temperature_k,36.0,356.584722,7.592106,342.4200,350.545000,356.5150,361.945000,369.5700
oil_rate_bpd,36.0,209.675000,59.467102,66.2000,172.300000,208.9000,254.000000,335.1000


In [3]:
screened = wells.loc[
    (wells["porosity_fraction"] >= 0.20)
    & (wells["water_saturation_fraction"] <= 0.35),
    ["well_id", "field", "porosity_fraction", "water_saturation_fraction", "permeability_md"],
].sort_values("permeability_md", ascending=False)
screened.head(8)


,well_id,field,porosity_fraction,water_saturation_fraction,permeability_md
9,W-10,South,0.2850,0.3425,64.90
20,W-21,North,0.2290,0.3438,58.29
1,W-02,North,0.2528,0.3116,42.40


In [4]:
field_summary = wells.groupby("field", as_index=False).agg(
    wells=("well_id", "count"),
    mean_porosity=("porosity_fraction", "mean"),
    median_permeability_md=("permeability_md", "median"),
    mean_oil_rate_bpd=("oil_rate_bpd", "mean"),
)
field_summary.round(3)


,field,wells,mean_porosity,median_permeability_md,mean_oil_rate_bpd
0,Central,12,0.205,50.035,212.700
1,North,12,0.218,45.470,214.358
2,South,12,0.216,52.370,201.967


## Challenge

        Add a permeability class, compare mean rate by field and class, then explain why this is
        descriptive rather than causal analysis.